# 01 — Data exploration and QC

Before any modelling: does the cohort look like cfDNA, and does anything
separate the groups for reasons that are not disease?

The QC questions worth answering here are the ones that would invalidate
everything downstream: are the size distributions nucleosomal, is depth
balanced across groups, and is any technical covariate label-correlated.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np, pandas as pd
from cfdna_insight.features import load_features

FEATURES = '../results/features'   # or '../results/smoke/features'
blocks = load_features(FEATURES)
samples = pd.read_csv(os.path.join(FEATURES, 'samples_used.tsv'), sep='\t')
order = [s for s in samples['sample_id'] if s in blocks['size_summary'].index]
samples = samples.set_index('sample_id').loc[order].reset_index()
y = (samples['label'] == 'cancer').astype(int).to_numpy()
{k: v.shape for k, v in blocks.items()}

## Cohort composition

Cross-tabulate label against every metadata column that exists. Any cell
that is empty or near-empty is a confounder waiting to be discovered later.

In [ ]:
for col in samples.columns:
    if col in ('sample_id', 'path'):
        continue
    print(pd.crosstab(samples['label'], samples[col]), '\n')

## Sequencing depth by group

If cancer samples were sequenced deeper, every count-based feature is
confounded. Check this before believing anything else.

In [ ]:
tech = blocks['technical'].reindex(order)
depth = tech['tech_n_fragments']
print(depth.groupby(samples['label'].to_numpy()).describe())

from scipy.stats import mannwhitneyu
u, p = mannwhitneyu(depth[y == 1], depth[y == 0])
print(f'\ndepth difference between groups: p = {p:.3g}')
if p < 0.05:
    print('WARNING: depth is label-correlated. Consider downsampling all',
          'samples to a common fragment count before feature extraction.')

## Every technical covariate against label

This is a preview of the negative control in notebook 03. Anything with a
small p here will show up there as a technical model that beats chance.

In [ ]:
rows = []
for col in tech.columns:
    v = tech[col].to_numpy(dtype=float)
    ok = np.isfinite(v)
    if ok.sum() < len(v) * 0.8 or len(np.unique(v[ok])) < 3:
        continue
    u, p = mannwhitneyu(v[ok & (y == 1)], v[ok & (y == 0)])
    rows.append({'covariate': col, 'p': p})
pd.DataFrame(rows).sort_values('p')

## Size distributions

Look for the 167 bp peak and the 10 bp ripple. A sample missing either is
degraded and is a candidate for exclusion — on QC grounds, decided before
looking at its label.

In [ ]:
import matplotlib.pyplot as plt
from cfdna_insight.plots import plot_size_distributions
plot_size_distributions(blocks['size_hist'].reindex(order), y, '/tmp/sizes.png')
from IPython.display import Image; Image('/tmp/sizes.png')

In [ ]:
summ = blocks['size_summary'].reindex(order)
flag = summ['periodicity_10bp'] < summ['periodicity_10bp'].quantile(0.05)
print('low-periodicity samples (possible degradation):')
print(samples.loc[flag.to_numpy(), ['sample_id', 'label', 'study']])